Coauthored and disputed - test set, not train
Train - hamilton and madison


In [1]:

# 1) Imports and paths
from pathlib import Path
import pandas as pd
from lexos.scrubber.scrubber import Scrubber
from lexos.tokenizer import Tokenizer
from lexos.dtm import DTM, Vectorizer
from sklearn.svm import SVC
from lexos.classification import trainer

BASE = Path.cwd()  # run this notebook from doc_src/docs/tutorials/classification
DATA = BASE / "fed_papers"

print("Base:", BASE)
print("Data dir exists:", DATA.exists())

Base: c:\Users\gabal\OneDrive\Lexos_Independant_Research\uv_lexos\doc_src\docs\tutorials\classification
Data dir exists: True


In [2]:
# 2) Collect train and test files
train_dirs = ["HAMILTON", "MADISON"]
test_dirs = ["COAUTHORED", "DISPUTED"]

train_files = []
for d in train_dirs:
    train_files.extend(sorted((DATA / d).glob("*.txt")))

test_files = []
for d in test_dirs:
    test_files.extend(sorted((DATA / d).glob("*.txt")))

print("Train files:", len(train_files), "Test files:", len(test_files))
assert train_files and test_files, "No files found; check working directory and folder structure."

Train files: 65 Test files: 15


In [3]:
# 3) Scrubber and tokenizer
scrubber = Scrubber()
scrubber.add_pipe("lower_case")
scrubber.add_pipe("digits")
scrubber.add_pipe("punctuation")

tokenizer = Tokenizer(model="en_core_web_sm")

In [4]:
# 4) Build training corpus (HAMILTON/MADISON)
train_token_lists, train_doc_ids, y_train = [], [], []

for f in train_files:
    raw = f.read_text(encoding="utf-8")
    clean = scrubber.scrub(raw)
    doc = tokenizer(clean)  # same as tokenizer.make_doc(clean)
    tokens = [t.text for t in doc if t.text.strip()]
    train_token_lists.append(tokens)
    train_doc_ids.append(f.name)       # row label for DTM (not used by sklearn)
    y_train.append(f.parent.name)      # target = HAMILTON or MADISON

print("Train docs:", len(train_token_lists), "Targets:", set(y_train))
assert len(train_token_lists) == len(train_doc_ids) == len(y_train) and len(train_token_lists) > 0

Train docs: 65 Targets: {'MADISON', 'HAMILTON'}


In [5]:
# 5) Build test corpus (COAUTHORED/DISPUTED) — labels here are group tags, not training classes
test_token_lists, test_doc_ids, test_groups = [], [], []

for f in test_files:
    raw = f.read_text(encoding="utf-8")
    clean = scrubber.scrub(raw)
    doc = tokenizer(clean)
    tokens = [t.text for t in doc if t.text.strip()]
    test_token_lists.append(tokens)
    test_doc_ids.append(f.name)
    test_groups.append(f.parent.name)  # COAUTHORED or DISPUTED

print("Test docs:", len(test_token_lists), "Groups:", set(test_groups))
assert len(test_token_lists) == len(test_doc_ids) == len(test_groups) and len(test_token_lists) > 0

Test docs: 15 Groups: {'DISPUTED', 'COAUTHORED'}


In [6]:
# 6) Vectorize: fit on TRAIN only, transform TEST with the same vocabulary
dtm_train = DTM(vectorizer=Vectorizer())
_ = dtm_train(docs=train_token_lists, labels=train_doc_ids)
X_train = dtm_train.doc_term_matrix

# Transform test using the fitted vectorizer
test_strings = [" ".join(toks) for toks in test_token_lists]
X_test = dtm_train.vectorizer.transform(test_strings)

print("Train DTM:", X_train.shape, "Vocab size:", len(dtm_train.sorted_terms_list))
print("Test DTM:", X_test.shape)

Train DTM: (65, 8001) Vocab size: 8001
Test DTM: (15, 8001)


In [ ]:
# 7) Train classifier on HAMILTON/MADISON and predict authorship for COAUTHORED/DISPUTED
clf = trainer.fit_classifier(X_train, y_train, model="svc", kernel="linear") 
# fit_classifier() is a new method that allows to train
# on an already split training set 
y_pred_test = clf.predict(X_test)

pred_df = pd.DataFrame({
    "doc_id": test_doc_ids,
    "set": test_groups,
    "pred_author": y_pred_test
}).sort_values(["set", "doc_id"])

display(pred_df)
print("\nPrediction counts by set:")
print(pred_df.groupby(["set", "pred_author"]).size())

,doc_id,set,pred_author
0,FED_18_C.txt,COAUTHORED,HAMILTON
1,FED_19_C.txt,COAUTHORED,HAMILTON
2,FED_20_C.txt,COAUTHORED,HAMILTON
3,FED_49_D.txt,DISPUTED,HAMILTON
4,FED_50_D.txt,DISPUTED,HAMILTON
5,FED_51_D.txt,DISPUTED,HAMILTON
6,FED_52_D.txt,DISPUTED,HAMILTON
7,FED_53_D.txt,DISPUTED,HAMILTON
8,FED_54_D.txt,DISPUTED,HAMILTON
9,FED_55_D.txt,DISPUTED,HAMILTON



Prediction counts by set:
set         pred_author
COAUTHORED  HAMILTON        3
DISPUTED    HAMILTON       12
dtype: int64
